In [ ]:
!pip install wikibaseintegrator # new line of code
import pandas as pd
import re
import time
from wikibaseintegrator.wbi_exceptions import MWApiError, ModificationFailed # 補上這一個
from wikibaseintegrator import WikibaseIntegrator, wbi_login
from wikibaseintegrator import datatypes
from wikibaseintegrator.models import Claims, Sitelinks
from wikibaseintegrator.wbi_config import config
from wikibaseintegrator.models.labels import Labels
from wikibaseintegrator.models.descriptions import Descriptions
from wikibaseintegrator.wbi_exceptions import MWApiError # Import MWApiError
import requests

In [ ]:
# 1: Write column Properties to Wikibase
# -----------------------------------------
# find all Pids
df = pd.read_csv('LOD Tablet Dictionary (FG Cuneiform) - museum_FG - LOD Tablet Dictionary (FG Cuneiform) - museum_FG.csv').drop(0)
properties_col_labels = [re.findall(r'P[0-9]+', col)[0] for col in df if col.startswith('P')]
# remove duplicates
properties = list(set(properties_col_labels))
print(properties_col_labels)

In [ ]:
# Sign in to create WikibaseIntegrator
# when run thru colab: run ngrok http 8181 in the terminal

# config['USER_AGENT'] = 'Property Import Script'
# config['MEDIAWIKI_API_URL'] = 'https://undepressing-vinita-spurtively.ngrok-free.dev'
# WDUSER_test = input("wikibase username")
# WDPASS_test = input("wikibase password")
# login_testwikidata = wbi_login.Login(user=WDUSER_test, password=WDPASS_test, mediawiki_api_url='http://localhost:8181/w/api.php')
config['USER_AGENT'] = 'Property Import Script'

my_ngrok_url = 'https://undepressing-vinita-spurtively.ngrok-free.dev'

WDUSER_test = input("wikibase username: ")
WDPASS_test = input("wikibase password: ")

login_testwikidata = wbi_login.Login(
    user=WDUSER_test,
    password=WDPASS_test,
    mediawiki_api_url=f'{my_ngrok_url}/w/api.php'
)

WDUSER = input("factgrid username")
WDPASS = input("factgrid password")
login_wikidata = wbi_login.Login(user=WDUSER, password=WDPASS, mediawiki_api_url='https://database.factgrid.de/w/api.php') #change to FactGrid's api

wbi = WikibaseIntegrator(login=login_testwikidata)

In [ ]:
#step2 (version 1)
# target_qid_dict = {}

# # 1. Identify all unique FactGrid QIDs used in the property columns (except P771)
# property_cols_for_mapping = df[[col for col in df if col.startswith('P') and col != 'P771']]
# qid_property_cols = df[[i for i in property_cols_for_mapping if (type(property_cols_for_mapping[i][1]) == str and property_cols_for_mapping[i][1].startswith('Q'))]]

# qids_list = []
# for col in qid_property_cols:
#     qids_list += (list(qid_property_cols[col].dropna().unique()))
# qids_list = list(set(qids_list))

# print(f"Mapping {len(qids_list)} target entities to local Wikibase...\n")

# for qid in qids_list:
#     time.sleep(1) # Delay to stabilize ngrok connection
#     try:
#         # Fetch label from FactGrid
#         source_entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)
#         en_label = str(source_entity.labels.get('en'))

#         # Search for this label in your local Wikibase
#         search_params = {
#             'action': 'wbsearchentities',
#             'search': en_label,
#             'language': 'en',
#             'type': 'item',
#             'format': 'json'
#         }
#         search_res = requests.get(f'{my_ngrok_url}/w/api.php', params=search_params).json()

#         if search_res.get('search'):
#             local_id = search_res['search'][0]['id']
#             target_qid_dict[qid] = [local_id]
#             print(f"🔗 Mapped existing: {qid} -> {local_id} ({en_label})")
#         else:
#             # If not found locally, create it
#             new_item = wbi.item.new()
#             new_item.labels.set(language='en', value=en_label)
#             # Add original description if available
#             source_desc = source_entity.descriptions.get('en')
#             if source_desc:
#                 new_item.descriptions.set(language='en', value=str(source_desc))

#             new_item.sitelinks = Sitelinks()
#             written_item = new_item.write(mediawiki_api_url=f'{my_ngrok_url}/w/api.php', login=login_testwikidata, as_new=True)
#             target_qid_dict[qid] = [written_item.id]
#             print(f"✅ Created new: {en_label} as {written_item.id}")

#     except Exception as e:
#         print(f"🛑 Error processing {qid}: {e}")

# print("\n--- target_qid_dict is ready ---")



#step2 (version 2) Build and Repair Target Entity Mapping ---
target_qid_dict = {}

# 1. Identify all unique FactGrid QIDs used in the property columns (excluding P771)
property_cols_for_mapping = df[[col for col in df if col.startswith('P') and col != 'P771']]
qid_property_cols = df[[i for i in property_cols_for_mapping if (type(property_cols_for_mapping[i][1]) == str and property_cols_for_mapping[i][1].startswith('Q'))]]

qids_list = []
for col in qid_property_cols:
    qids_list += (list(qid_property_cols[col].dropna().unique()))
qids_list = list(set(qids_list))

print(f"🔍 Checking and mapping {len(qids_list)} target entities (cities, orgs, etc.) to local Wikibase...\n")

for qid in qids_list:
    time.sleep(2)  # Increased delay to stabilize ngrok and avoid 503 errors
    try:
        # Fetch the original label from FactGrid
        source_entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)
        en_label = str(source_entity.labels.get('en'))

        if not en_label or en_label == 'None':
            en_label = f"Entity {qid}"

        # 2. Search if this label already exists in your local Wikibase
        search_params = {
            'action': 'wbsearchentities',
            'search': en_label,
            'language': 'en',
            'type': 'item',
            'format': 'json'
        }
        search_res = requests.get(f'{my_ngrok_url}/w/api.php', params=search_params).json()

        if search_res.get('search'):
            # If it exists locally, just record the mapping (prevents duplicates)
            local_id = search_res['search'][0]['id']
            target_qid_dict[qid] = [local_id]
            print(f"🔗 Mapped existing: {qid} -> {local_id} ({en_label})")
        else:
            # If not found locally, create a new item
            new_item = wbi.item.new()
            new_item.labels.set(language='en', value=en_label)

            # Fetch original description if available
            source_desc = source_entity.descriptions.get('en')
            if source_desc:
                new_item.descriptions.set(language='en', value=str(source_desc))

            new_item.sitelinks = Sitelinks()
            written_item = new_item.write(mediawiki_api_url=f'{my_ngrok_url}/w/api.php', login=login_testwikidata, as_new=True)

            target_qid_dict[qid] = [written_item.id]
            print(f"✅ Created new: {en_label} as {written_item.id} (FactGrid: {qid})")

    except Exception as e:
        print(f"🛑 Error processing {qid}: {e}")

print("\n✨ --- target_qid_dict is now complete --- ✨")

In [ ]:
property_dict

In [ ]:
# # --- Step 3 (version1): Final Museum Entry Import (Language Cleanup Version) ---
# source_qids_list = list(df['qid'].dropna().unique())

# print(f"Starting final import for {len(source_qids_list)} museum entries...")

# for qid in source_qids_list:
#     time.sleep(3.5)
#     try:
#         source_entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)
#         # Force label to string
#         en_label = str(source_entity.labels.get('en'))
#         en_desc = str(source_entity.descriptions.get('en')) if source_entity.descriptions.get('en') else ""

#         if not en_label or en_label == 'None':
#              en_label = f"Museum Entry {qid}"
#     except Exception as e:
#         print(f"❌ Could not fetch source {qid}: {e}")
#         continue

#     # 1. Search locally
#     local_qid = None
#     search_params = {
#         'action': 'wbsearchentities',
#         'search': en_label,
#         'language': 'en',
#         'type': 'item',
#         'format': 'json'
#     }
#     try:
#         search_res = requests.get(f'{my_ngrok_url}/w/api.php', params=search_params).json()
#         if search_res.get('search'):
#             local_qid = search_res['search'][0]['id']
#             print(f"🔍 Found existing: {en_label} -> {local_qid}")
#     except:
#         pass

#     # 2. CRITICAL FIX: Always start with a NEW local entity object
#     # This prevents carrying over unrecognized language codes from the database
#     local_entity = wbi.item.new()
#     if local_qid:
#         local_entity.id = local_qid # Set ID to trigger an UPDATE instead of a CREATE

#     # 3. ONLY set the languages your Wikibase supports (English)
#     local_entity.labels.set(language='en', value=en_label)
#     if en_desc:
#         local_entity.descriptions.set(language='en', value=en_desc)

#     local_entity.claims = Claims()
#     local_entity.sitelinks = Sitelinks()

#     # 4. Map Claims from CSV
#     entry_claims = df[df['qid'] == qid].iloc[0]
#     for col in df.columns:
#         if col.startswith('P') and col != 'P287':
#             source_pid = re.findall(r'P[0-9]+', col)[0]
#             val = entry_claims[col]

#             if pd.isna(val) or source_pid not in property_dict:
#                 continue

#             local_pid = property_dict[source_pid]

#             try:
#                 if str(val).startswith('Q'):
#                     if val in target_qid_dict and target_qid_dict[val]:
#                         local_target_qid = target_qid_dict[val][0]
#                         local_entity.claims.add(datatypes.Item(prop_nr=local_pid, value=local_target_qid))
#                 elif str(val).startswith('http'):
#                     local_entity.claims.add(datatypes.URL(prop_nr=local_pid, value=str(val)))
#                 elif source_pid == 'P771':
#                     local_entity.claims.add(datatypes.ExternalID(prop_nr=local_pid, value=str(val)))
#             except:
#                 pass

#     # 5. Write (Update if local_qid exists, otherwise Create)
#     try:
#         # Use as_new=False if we found a local_qid
#         final_item = local_entity.write(
#             mediawiki_api_url=f'{my_ngrok_url}/w/api.php',
#             login=login_testwikidata,
#             as_new=not bool(local_qid)
#         )
#         status = "Updated" if local_qid else "Created"
#         print(f"✅ {status}: {en_label} as {final_item.id}")
#     except Exception as e:
#         print(f"🛑 Failed to save {en_label}: {e}")



# --- Step 3 (version2)
# source_qids_list = list(df['qid'].dropna().unique())

# print(f"Starting final import for {len(source_qids_list)} museum entries...")

# for qid in source_qids_list:
#     # 🌟 Increase the delay to 4 seconds to ensure that the ngrok tunnel and local server are not overloaded.
#     time.sleep(4)

#     try:
#         source_entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)
#         en_label = str(source_entity.labels.get('en'))
#         en_desc = str(source_entity.descriptions.get('en')) if source_entity.descriptions.get('en') else ""

#         if not en_label or en_label == 'None':
#              en_label = f"Museum Entry {qid}"
#     except Exception as e:
#         print(f"❌ Could not fetch source {qid}: {e}")
#         continue

#     # 1. Search if it already exists locally
#     local_qid = None
#     search_params = {
#         'action': 'wbsearchentities',
#         'search': en_label,
#         'language': 'en',
#         'type': 'item',
#         'format': 'json'
#     }
#     try:
#         search_res = requests.get(f'{my_ngrok_url}/w/api.php', params=search_params).json()
#         if search_res.get('search'):
#             local_qid = search_res['search'][0]['id']
#             print(f"🔍 Found existing: {en_label} -> {local_qid}")
#     except:
#         pass

#     # 2.Create a new item or prepare to update an old item
#     local_entity = wbi.item.new()
#     if local_qid:
#         local_entity.id = local_qid

#     local_entity.labels.set(language='en', value=en_label)
#     if en_desc:
#         local_entity.descriptions.set(language='en', value=en_desc)

#     local_entity.claims = Claims()
#     local_entity.sitelinks = Sitelinks()

#     # 3. Map Claims (plus QID security checks)
#     entry_claims = df[df['qid'] == qid].iloc[0]
#     for col in df.columns:
#         if col.startswith('P') and col != 'P287':
#             source_pid = re.findall(r'P[0-9]+', col)[0]
#             val = entry_claims[col]

#             if pd.isna(val) or source_pid not in property_dict:
#                 continue

#             local_pid = property_dict[source_pid]

#             try:
#                 if str(val).startswith('Q'):
#                     # 🌟 Core Fix: Added check to avoid KeyError (resolves the Q21411 issue you just encountered)
#                     if val in target_qid_dict and target_qid_dict[val]:
#                         local_target_qid = target_qid_dict[val][0]
#                         local_entity.claims.add(datatypes.Item(prop_nr=local_pid, value=local_target_qid))
#                     else:
#                         print(f"⚠️ Skipping link: Target {val} not found in dictionary (for {en_label})")

#                 elif str(val).startswith('http'):
#                     local_entity.claims.add(datatypes.URL(prop_nr=local_pid, value=str(val)))

#                 elif source_pid == 'P771':
#                     local_entity.claims.add(datatypes.ExternalID(prop_nr=local_pid, value=str(val)))
#             except Exception as claim_e:
#                 print(f"⚠️ Claim Error: {claim_e}")

#     # 4. write into local
#     try:
#         final_item = local_entity.write(
#             mediawiki_api_url=f'{my_ngrok_url}/w/api.php',
#             login=login_testwikidata,
#             as_new=not bool(local_qid)
#         )
#         status = "Updated" if local_qid else "Created"
#         print(f"✅ {status}: {en_label} as {final_item.id}")
#     except Exception as e:
#         print(f"🛑 Failed to save {en_label}: {e}")


# # step 3 (version 3): Write Entity (sources) + Claims (Property + target) to Wikibasement
# source_qids_list = list(df['qid'].dropna().unique())

# for i in range(len(source_qids_list)):
#     qid = source_qids_list[i]

#     # Fetch original entity from FactGrid
#     entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)
#     entity.claims = Claims()
#     entity.sitelinks = Sitelinks() # Clear sitelinks to avoid "Unknown site" errors

#     # Filter claims for this specific museum
#     entry_claims = df[df['qid'] == qid]
#     entry_claims = entry_claims[[col for col in entry_claims if col.startswith('P') and col != 'P287']].iloc[0]

#     for item_idx in range(len(entry_claims)):
#         val = entry_claims.iloc[item_idx] # Using .iloc to avoid FutureWarning
#         source_pid = properties_col_labels[item_idx]

#         if pd.isna(val):
#             continue

#         new_pid = property_dict.get(source_pid)
#         if not new_pid:
#             continue

#         # 4. Search and Write (Update logic)
#     try:
#         # Check if this item already exists locally to get its ID
#         search_params = {
#             'action': 'wbsearchentities',
#             'search': str(entity.labels.get('en')),
#             'language': 'en',
#             'type': 'item',
#             'format': 'json'
#         }
#         search_res = requests.get(f'{my_ngrok_url}/w/api.php', params=search_params).json()

#         local_id = None
#         if search_res.get('search'):
#             local_id = search_res['search'][0]['id']
#             entity.id = local_id # Set the ID so WBI knows to UPDATE Q926 instead of creating a new one
#             print(f"🔍 Found existing: {local_id}. Preparing update...")

#         # Write to local Wikibase
#         # as_new=True ONLY if local_id is None
#         final_item = entity.write(
#             mediawiki_api_url=f'{my_ngrok_url}/w/api.php',
#             login=login_testwikidata,
#             as_new=not bool(local_id)
#         )

#         status = "Updated" if local_id else "Created"
#         print(f"✅ {status}: {qid} as {final_item.id}")

#     except Exception as e:
#         print(f"🛑 Failed to write {qid}: {e}")


# --- Fixed Step 3 (version 4): Write Entity Claims with Safety Checks ---(Ver:)
# source_qids_list = list(df['qid'].dropna().unique())

# for i in range(len(source_qids_list)):
#     qid = source_qids_list[i]

#     # 1. Fetch original entity from FactGrid
#     try:
#         entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)
#         entity.claims = Claims()
#         entity.sitelinks = Sitelinks() # Avoid "Unknown site" errors
#     except Exception as e:
#         print(f"❌ Could not fetch {qid}: {e}")
#         continue

#     # 2. Filter claims from CSV for this specific museum
#     entry_claims_row = df[df['qid'] == qid]
#     if entry_claims_row.empty:
#         continue
#     entry_claims = entry_claims_row[[col for col in entry_claims_row if col.startswith('P') and col != 'P287']].iloc[0]

#     for item_idx in range(len(entry_claims)):
#         val = entry_claims.iloc[item_idx] # Fixes FutureWarning
#         source_pid = properties_col_labels[item_idx]

#         if pd.isna(val) or source_pid not in property_dict:
#             continue

#         local_pid = property_dict[source_pid]

#         try:
#             # Handle External IDs (P771)
#             if source_pid == 'P771':
#                 entity.claims.add(datatypes.ExternalID(prop_nr=local_pid, value=str(val)))

#             # Handle Item Links (QIDs)
#             elif str(val).startswith('Q') and source_pid != 'P771':
#                 # --- CRITICAL SAFETY CHECK: Fixes KeyError ---
#                 if val in target_qid_dict and target_qid_dict[val]:
#                     target_new_id = target_qid_dict[val][0]
#                     entity.claims.add(datatypes.Item(prop_nr=local_pid, value=target_new_id))
#                 else:
#                     print(f"⚠️ Skipping missing target: {val} (referenced by {qid})")

#             # Handle URLs
#             elif str(val).startswith('http'):
#                 entity.claims.add(datatypes.URL(prop_nr=local_pid, value=str(val)))

#         except Exception as e:
#             print(f"🛑 Error processing property {source_pid} for {qid}: {e}")

#     # 3. Update Existing Local Item (Prevents duplicate label errors)
#     try:
#         search_params = {'action': 'wbsearchentities', 'search': str(entity.labels.get('en')), 'language': 'en', 'type': 'item', 'format': 'json'}
#         search_res = requests.get(f'{my_ngrok_url}/w/api.php', params=search_params).json()

#         local_id = search_res['search'][0]['id'] if search_res.get('search') else None
#         if local_id:
#             entity.id = local_id # Tells WBI to UPDATE instead of CREATE

#         final_item = entity.write(
#             mediawiki_api_url=f'{my_ngrok_url}/w/api.php',
#             login=login_testwikidata,
#             as_new=not bool(local_id)
#         )
#         print(f"✅ {'Updated' if local_id else 'Created'}: {qid} as {final_item.id}")
#     except Exception as e:
#         print(f"🛑 Failed to write {qid}: {e}")

#
#     # break


# --- Final Step 3 (version 5): Optimized Museum Entry Import ---
source_qids_list = list(df['qid'].dropna().unique())

print(f"🚀 Starting final import for {len(source_qids_list)} museum entries...")

for i in range(len(source_qids_list)):
    qid = source_qids_list[i]

    # 1. Fetch original entity from FactGrid
    try:
        # We fetch the full item to get Labels and Descriptions automatically
        entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)

        # 🌟 FIX: Clear sitelinks to prevent 'Unknown site: enwiki' error in local Wikibase
        entity.sitelinks = Sitelinks()

        # Reset claims to start fresh with local property mappings
        entity.claims = Claims()
    except Exception as e:
        print(f"❌ Could not fetch source {qid} from FactGrid: {e}")
        continue

    # 2. Map Claims from your CSV
    entry_claims_row = df[df['qid'] == qid]
    if entry_claims_row.empty:
        continue

    # Get the specific row for this museum, excluding metadata columns
    entry_claims = entry_claims_row[[col for col in entry_claims_row if col.startswith('P') and col != 'P287']].iloc[0]

    for item_idx in range(len(entry_claims)):
        # 🌟 FIX: Use .iloc to avoid FutureWarning red lines
        val = entry_claims.iloc[item_idx]
        source_pid = properties_col_labels[item_idx]

        if pd.isna(val) or source_pid not in property_dict:
            continue

        local_pid = property_dict[source_pid]

        try:
            # Add Claims based on data type
            if source_pid == 'P771': # External ID logic
                entity.claims.add(datatypes.ExternalID(prop_nr=local_pid, value=str(val)))
            elif str(val).startswith('Q') and source_pid != 'P771': # Item/Link logic
                if val in target_qid_dict and target_qid_dict[val]:
                    local_target_qid = target_qid_dict[val][0]
                    entity.claims.add(datatypes.Item(prop_nr=local_pid, value=local_target_qid))
            elif str(val).startswith('http'): # URL logic
                entity.claims.add(datatypes.URL(prop_nr=local_pid, value=str(val)))
        except Exception as claim_e:
            print(f"⚠️ Claim Error on {source_pid} for {qid}: {claim_e}")

    # 3. Write to local Wikibase (Search first to avoid Label conflicts)
    try:
        # Check if the museum already exists locally by searching its English label
        search_params = {
            'action': 'wbsearchentities',
            'search': str(entity.labels.get('en')),
            'language': 'en',
            'type': 'item',
            'format': 'json'
        }
        search_res = requests.get(f'{my_ngrok_url}/w/api.php', params=search_params).json()

        local_id = search_res['search'][0]['id'] if search_res.get('search') else None

        if local_id:
            entity.id = local_id # Tells the system to UPDATE the existing item

        final_item = entity.write(
            mediawiki_api_url=f'{my_ngrok_url}/w/api.php',
            login=login_testwikidata,
            as_new=not bool(local_id) # True if creating new, False if updating
        )
        print(f"✅ Processed: {qid} -> Local {final_item.id} ({entity.labels.get('en')})")

    except Exception as write_e:
        print(f"🛑 Write Error on {qid}: {write_e}")

    # REMOVE THE BREAK BELOW to process all 109 entries once the first test is successful


In [ ]:
target_qid_dict
# qids_list

In [ ]:
new_properties_list

In [ ]:
property_dict

In [ ]:
property_cols = df[[col for col in df if col.startswith('P')]]
property_cols

In [ ]:
# property_dict
# # properties_col_labels
# properties_col_labels.remove('P287')
# properties_col_labels

# Check if P287 exists before removing to avoid ValueError
if 'P287' in properties_col_labels:
    properties_col_labels.remove('P287')
    print("Successfully removed P287")
else:
    print("P287 was not in the list or already removed")

# Display the resulting dictionary and list
print("Current Property Dictionary:", property_dict)
print("Updated Property Column Labels:", properties_col_labels)

In [ ]:
entry_claims

In [ ]:
# 3: Write Entity (sources) + Claims (Property + target) to Wikibasement
source_qids_list = list(df['qid'].dropna().unique())
for i in range(len(source_qids_list)):
    qid = source_qids_list[i]
    entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)
    entity.claims = Claims()
    entry_claims = df[df['qid'] == qid]
    entry_claims = entry_claims[[col for col in entry_claims if col.startswith('P') and col != 'P287']].iloc[0]
    for item in range(len(entry_claims)):
        new_pid = property_dict[properties_col_labels[item]]
        if type(entry_claims[item]) == float:
            continue

        elif properties_col_labels[item] == 'P771':
            # target is original QID in Wikidata
            target = entry_claims[item]
            claim = datatypes.ExternalID(prop_nr=new_pid, value = target)
            entity.claims.add(claim)

        elif entry_claims[item].startswith('Q') and properties_col_labels[item] != 'P771':
            target_new_id = target_qid_dict[entry_claims[item]][0]
            claim = datatypes.Item(prop_nr=new_pid, value = target_new_id)
            entity.claims.add(claim)
        elif entry_claims[item].startswith('http'):
            target = entry_claims[item]
            print(new_pid)
            print(target)
            claim = datatypes.URL(prop_nr=new_pid, value = target)
            entity.claims.add(claim)
    #     # elif type(entry_claims[item]) == str:
    #     #     target = entry_claims[item]
    #     #     claim = datatypes.String(prop_nr=new_pid, value = target)
    #     #     entity.claims.add(claim)
    entity.write(mediawiki_api_url=f'{my_ngrok_url}/w/api.php', login=login_testwikidata, as_new=True)
    break



# target = entry_claims[3]
# new_pid = property_dict[properties_col_labels[3]]
# claim = datatypes.URL(prop_nr=new_pid, value = target)
# entity.claims.add(claim)
# entity.write(mediawiki_api_url='http://localhost/w/api.php', login=login_testwikidata, as_new=True)
entry_claims

In [ ]:
qid = source_qids_list[21]

entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)
entity.claims = Claims()
qid

In [ ]:
entry_claims = df[df['qid'] == qid]
entry_claims = entry_claims[[col for col in entry_claims if col.startswith('P')]].iloc[0]
entry_claims

In [ ]:

target_new_id = target_qid_dict[entry_claims[2]][0]
target_new_id

In [ ]:
new_pid = property_dict[properties_col_labels[2]]

properties_col_labels
new_pid


In [ ]:

claim = datatypes.Item(prop_nr=new_pid, value = target_new_id)
entity.claims.add(claim)

In [ ]:
entity

In [ ]:
entity.write(mediawiki_api_url=f'{my_ngrok_url}/w/api.php', login=login_testwikidata, as_new=True)

In [ ]:
#qid ='Q102010'
pid = 'P156'
target = 'https://www.britishmuseum.org/'
entity = wbi.item.get(qid, mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata)

claim = datatypes.URL(prop_nr=pid, value = target)
entity.claims.add(claim)
#entity.write(mediawiki_api_url='https://database.factgrid.de/w/api.php', login=login_wikidata, as_new=False)